# Multi-Connection Hopfield Neural Network (MC-HNN) untuk Rekonstruksi Sidik Jari
Notebook ini berisi implementasi algoritma MC-HNN berdasarkan paper Jay Kant Pratap Singh Yadav, dkk (2022).

# 1. Load Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')
zip_path = "/content/drive/MyDrive/JST_Hopfield-Fingerprint/dataset_final.zip"
extract_path = "/content/dataset_final"

if not os.path.exists(extract_path):
    os.makedirs(extract_path)
    !unzip -q "{zip_path}" -d "{extract_path}"
    print("Dataset berhasil diekstrak ke lokal Colab!")
else:
    print("Dataset sudah tersedia.")

# 2. Define Networks (MCHNN) & Implementation

In [ ]:
import os
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt

class MCHNN:
    def __init__(self):
        self.etalon_arrays = []
        self.patterns = [] 
        self.E_vals = []
        
    def train(self, target_patterns):
        """
        Fase Pembelajaran (Learning Phase) MC-HNN
        target_patterns: List dari vektor bipolar 1D (ukuran n)
        """
        print(f"Memulai pelatihan untuk {len(target_patterns)} pola target...")
        for idx, p in enumerate(target_patterns):
            n = len(p)
            
            # 1. Hitung Etalon Array W_ij = s_i * s_j untuk i != j, dan W_ii = 0
            W = np.outer(p, p)
            np.fill_diagonal(W, 0)
            self.etalon_arrays.append(W)
            self.patterns.append(p)
            
            # 2. Hitung Energi Lyapunov untuk pola yang disimpan E(p)
            # E(p) = -0.5 * sum_i sum_{j!=i} s_i s_j w_ij
            E = -0.5 * np.sum(np.outer(p, p) * W)
            self.E_vals.append(E)
            print(f"Pola {idx+1} disimpan. Energi E(p) = {E}")

    def test(self, test_pattern, max_iter=20):
        """
        Fase Konvergensi (Convergence Phase) MC-HNN
        test_pattern: vektor bipolar 1D (citra noisy)
        """
        n = len(test_pattern)
        SE_vals = []
        
        # 3. Hitung SE(p) untuk input pattern terhadap masing-masing etalon array
        for W in self.etalon_arrays:
            SE = -0.5 * np.sum(np.outer(test_pattern, test_pattern) * W)
            SE_vals.append(SE)
            
        # 6. Cari affinity / Energi Minimum
        min_SE = min(SE_vals)
        candidate_indices = [i for i, val in enumerate(SE_vals) if val == min_SE]
        
        # 8. Gunakan Hamming Distance sebagai tie-breaker jika ada lebih dari satu kandidat
        if len(candidate_indices) > 1:
            hd_vals = []
            for idx in candidate_indices:
                hd = np.sum(test_pattern != self.patterns[idx])
                hd_vals.append(hd)
            best_idx = candidate_indices[np.argmin(hd_vals)]
        else:
            best_idx = candidate_indices[0]
            
        W_best = self.etalon_arrays[best_idx]
        E_best_stored = self.E_vals[best_idx]
        target_pattern = self.patterns[best_idx]
        
        # 4. Asynchronous update untuk merekonstruksi output
        O = test_pattern.copy()
        
        for iteration in range(max_iter):
            O_prev = O.copy()
            
            # Asynchronous random order
            indices = np.arange(n)
            np.random.shuffle(indices)
            
            for i in indices:
                # Onet_i(p) = I_i + sum_j O_j W_ij^p
                onet_i = test_pattern[i] + np.dot(O, W_best[i])
                if onet_i > 0:
                    O[i] = 1
                elif onet_i < 0:
                    O[i] = -1
                else:
                    O[i] = 0
                    
            # Jika sudah konvergen (tidak ada perubahan iterasi ini), berhenti
            if np.array_equal(O, O_prev):
                break
                
        # 5. Hitung SE1(p) dari pola output
        SE1 = -0.5 * np.sum(np.outer(O, O) * W_best)
        
        # 9. Cek stabilitas (Apakah pola yang direkonstruksi stabil?)
        is_stable = (SE1 == E_best_stored)
        
        return O, target_pattern, is_stable, best_idx

def preprocess_image(img_path, target_size=(30, 30)):
    """
    Melakukan preprocessing: Load citra, Equalization, Resize, Binarization, dan Bipolar mapping.
    """
    # Load grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Gagal membaca citra: {img_path}")
        return None, None
        
    # 1. Histogram Equalization
    img_eq = cv2.equalizeHist(img)
    
    # 2. Resize ROI (sesuai paper untuk mempercepat komputasi ke 30x30)
    img_resized = cv2.resize(img_eq, target_size)
    
    # 3. Binarization (Menggunakan metode adaptif / Otsu's Thresholding)
    _, img_bin = cv2.threshold(img_resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 4. Konversi ke pola Bipolar (-1, 1)
    bipolar_pattern = np.where(img_bin == 255, 1, -1).flatten()
    
    return bipolar_pattern, img_resized

def main():
    # PATH DATASET
    DATASET_PATH = '/content/dataset_asli'
    
    print(f"Mencari dataset di: {DATASET_PATH}...")
    
    # Data Selection: Pola Target diakhiri '_1.tif'
    target_files = glob.glob(os.path.join(DATASET_PATH, '*_1.tif'))
    all_files = glob.glob(os.path.join(DATASET_PATH, '*.tif'))
    test_files = [f for f in all_files if f not in target_files]
    
    if not target_files:
        print("PERINGATAN: Tidak ada file '_1.tif' yang ditemukan. Pastikan dataset sudah diekstrak dengan benar di /content/dataset_asli.")
        # Untuk tujuan demonstrasi bila tidak dijalankan di colab
        # return
        
    print(f"Ditemukan {len(target_files)} pola target (Clean).")
    print(f"Ditemukan {len(test_files)} pola uji (Noisy).")
    
    # 1. Preprocessing dan penyimpanan Pola Target
    target_patterns = []
    target_images = [] # Untuk visualisasi
    
    for f in target_files:
        bipolar, img_resized = preprocess_image(f)
        if bipolar is not None:
            target_patterns.append(bipolar)
            target_images.append(img_resized)
            
    # 2. Inisialisasi dan Training MC-HNN
    mc_hnn = MCHNN()
    if target_patterns:
        mc_hnn.train(target_patterns)
    else:
        print("Tidak ada pola yang bisa dilatih. Program berhenti.")
        return

    # 3. Pengujian dan Visualisasi
    print("\n--- Memulai Fase Pengujian (Rekonstruksi Citra) ---")
    for test_file in test_files:
        bipolar_test, img_test_resized = preprocess_image(test_file)
        if bipolar_test is None:
            continue
            
        print(f"\nMenguji citra: {os.path.basename(test_file)}")
        
        # Uji ke MC-HNN
        output_pattern, recognized_target, is_stable, recognized_idx = mc_hnn.test(bipolar_test)
        
        # Konversi output 1D bipolar kembali ke 2D untuk divisualisasikan
        # Mapping 1 ke putih (255) dan -1 ke hitam (0)
        recon_img = np.where(output_pattern == 1, 255, 0).reshape((30, 30))
        target_img = np.where(recognized_target == 1, 255, 0).reshape((30, 30))
        noisy_img = np.where(bipolar_test == 1, 255, 0).reshape((30, 30))
        
        # Visualisasi dengan Matplotlib (Satu baris: Asli, Noisy, Rekonstruksi)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Citra Asli (Pola Target yang paling mirip)
        axes[0].imshow(target_img, cmap='gray')
        axes[0].set_title(f"Citra Asli (Target #{recognized_idx+1})")
        axes[0].axis('off')
        
        # Citra Noisy (Input)
        axes[1].imshow(noisy_img, cmap='gray')
        axes[1].set_title("Citra Noisy")
        axes[1].axis('off')
        
        # Citra Hasil Rekonstruksi
        status_teks = "Konvergen Stabil" if is_stable else "Tidak Stabil"
        axes[2].imshow(recon_img, cmap='gray')
        axes[2].set_title(f"Citra Hasil Rekonstruksi\n({status_teks})")
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

if __name__ == '__main__':
    main()

